# Steam Review Humor: Experiment-Focused Data Exploration

This notebook is organized for your seminar setup (prompt-based BERT MLM with base vs fine-tuned models, 5 prompts, and 2 input text variants):

1. Dataset quality and coverage.
2. Label-space imbalance and per-prompt training capacity.
3. Input-length/tokenization effects (including truncation risk at `max_length=256`).
4. Game-level and temporal variation in humor labels.
5. Signals that can influence MLM prompt behavior (votes, playtime, lexical overlap with label words).


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from transformers import BertTokenizer, logging as hf_logging

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120
pd.options.display.max_columns = 200
pd.options.display.float_format = "{:.4f}".format

DATA_PATH = Path("./data/preprocessed/steam_reviews_preprocessed.csv")
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256

PROMPT_SPECS = [
    {
        "id": "decimal_0",
        "prompt": "On a scale from 0.0 to 1.0 this review is 0.[MASK] funny.",
        "label_words": ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"],
        "label_col": "label_funny_minmax",
    },
    {
        "id": "decimal_1",
        "prompt": "On a scale from 0 to 1 this review is 0.[MASK] funny.",
        "label_words": ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"],
        "label_col": "label_funny_minmax",
    },
    {
        "id": "words_0",
        "prompt": "Overall, the humor is [MASK].",
        "label_words": ["serious", "witty", "amusing", "hilarious", "hysterical"],
        "label_col": "label_votes_funny_categorical",
    },
    {
        "id": "words_1",
        "prompt": "Overall the humor of this review is [MASK].",
        "label_words": ["serious", "witty", "amusing", "hilarious", "hysterical"],
        "label_col": "label_votes_funny_categorical",
    },
    {
        "id": "words_2",
        "prompt": "The humor in this review is [MASK].",
        "label_words": ["serious", "witty", "amusing", "hilarious", "hysterical"],
        "label_col": "label_votes_funny_categorical",
    },
]

TEXT_COLUMNS = {
    "review_only": "review_text_cleaned",
    "review_plus_game": "review_info_text_cleaned",
}


In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {DATA_PATH.resolve()}")

USECOLS = [
    "review_id",
    "review_text_cleaned",
    "review_info_text_cleaned",
    "review_timestamp_created",
    "review_votes_funny",
    "review_votes_up",
    "review_weighted_vote_score",
    "review_comment_count",
    "review_steam_purchase",
    "author_playtime_forever",
    "author_playtime_at_review",
    "app_name",
    "app_genres",
    "app_categories",
    "app_release_date",
    "label_is_funny_binary",
    "label_funny_minmax",
    "label_votes_funny_categorical",
]

df = pd.read_csv(
    DATA_PATH,
    usecols=USECOLS,
    parse_dates=["review_timestamp_created"],
)

df["app_release_date"] = pd.to_datetime(
    df["app_release_date"],
    format="%d %b, %Y",
    errors="coerce",
    utc=True,
)

for col in ["app_name", "app_genres", "app_categories"]:
    df[col] = df[col].astype("category")

overview = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique_games",
            "review_period_start",
            "review_period_end",
            "median_review_age_days",
            "memory_usage_mb",
        ],
        "value": [
            f"{len(df):,}",
            df.shape[1],
            df["app_name"].nunique(),
            df["review_timestamp_created"].min(),
            df["review_timestamp_created"].max(),
            int((pd.Timestamp.utcnow() - df["review_timestamp_created"].median()).days),
            round(df.memory_usage(deep=True).sum() / 1024 / 1024, 1),
        ],
    }
)

display(overview)

display(df.head(3))


,metric,value
0,rows,"722,790"
1,columns,18
2,unique_games,23
3,review_period_start,2015-11-11 11:07:15+00:00
4,review_period_end,2026-01-13 00:59:31+00:00
5,median_review_age_days,235
6,memory_usage_mb,1074.0000


,review_id,review_text_cleaned,review_info_text_cleaned,review_timestamp_created,review_votes_up,review_votes_funny,review_weighted_vote_score,review_comment_count,review_steam_purchase,author_playtime_forever,author_playtime_at_review,app_name,app_categories,app_genres,app_release_date,label_is_funny_binary,label_funny_minmax,label_votes_funny_categorical
0,205748843,I don’t write this lightly. Destiny 2 has been...,I don’t write this lightly. Destiny 2 has been...,2025-10-03 06:23:08+00:00,864,26,0.9653,0,True,446776,435319,Destiny 2,"['Single-player', 'Multi-player', 'PvP', 'Onli...","['Action', 'Adventure', 'Free To Play']",2019-10-01 00:00:00+00:00,1,0.2000,4
1,203473553,"First of all let me say this, I have been a lo...","First of all let me say this, I have been a lo...",2025-09-04 00:00:19+00:00,967,51,0.9420,0,True,614677,605908,Destiny 2,"['Single-player', 'Multi-player', 'PvP', 'Onli...","['Action', 'Adventure', 'Free To Play']",2019-10-01 00:00:00+00:00,1,0.5000,4
2,199599201,The vast majority of the content I have paid f...,The vast majority of the content I have paid f...,2025-07-12 07:25:34+00:00,697,13,0.9413,92,True,99501,99501,Destiny 2,"['Single-player', 'Multi-player', 'PvP', 'Onli...","['Action', 'Adventure', 'Free To Play']",2019-10-01 00:00:00+00:00,1,0.1000,4


## 0) Loaded Dataset Profile

Quick sanity checks on the loaded preprocessed data: vote/date extremes, review text length range, and game taxonomy coverage (categories/genres).


In [3]:
def _explode_list_like(series):
    values = series.astype("string").fillna("")

    # Primary format in this dataset: "['A', 'B', ...]"
    parsed = values.str.findall(r"'([^']+)'")

    # Fallback for other separators if any appear (e.g., "A|B" or "A;B").
    fallback_mask = values.ne("") & parsed.str.len().fillna(0).eq(0)
    if fallback_mask.any():
        parsed.loc[fallback_mask] = (
            values.loc[fallback_mask]
            .str.split(r"\s*[|;]\s*")
            .apply(lambda parts: [p.strip() for p in parts if p and p.strip()])
        )

    return parsed.explode().dropna()

review_text_chars = df["review_text_cleaned"].fillna("").str.len()
review_text_words = df["review_text_cleaned"].fillna("").str.count(r"\S+")
review_info_text_chars = df["review_info_text_cleaned"].fillna("").str.len()
review_info_text_words = df["review_info_text_cleaned"].fillna("").str.count(r"\S+")

dataset_profile = pd.DataFrame(
    {
        "metric": [
            "review_votes_funny_min",
            "review_votes_funny_max",
            "review_timestamp_min",
            "review_timestamp_max",
            "review_text_cleaned_chars_min",
            "review_text_cleaned_chars_max",
            "review_text_cleaned_chars_median",
            "review_text_cleaned_words_min",
            "review_text_cleaned_words_max",
            "review_text_cleaned_words_median",
            "review_info_text_cleaned_chars_min",
            "review_info_text_cleaned_chars_max",
            "review_info_text_cleaned_chars_median",
            "review_info_text_cleaned_words_min",
            "review_info_text_cleaned_words_max",
            "review_info_text_cleaned_words_median",
        ],
        "value": [
            df["review_votes_funny"].min(),
            df["review_votes_funny"].max(),
            df["review_timestamp_created"].min(),
            df["review_timestamp_created"].max(),
            int(review_text_chars.min()),
            int(review_text_chars.max()),
            float(review_text_chars.median()),
            int(review_text_words.min()),
            int(review_text_words.max()),
            float(review_text_words.median()),
            int(review_info_text_chars.min()),
            int(review_info_text_chars.max()),
            float(review_info_text_chars.median()),
            int(review_info_text_words.min()),
            int(review_info_text_words.max()),
            float(review_info_text_words.median()),
        ],
    }
)

game_meta = df[["app_name", "app_categories", "app_genres"]].drop_duplicates(subset=["app_name"])
category_counts = _explode_list_like(game_meta["app_categories"]).value_counts()
genre_counts = _explode_list_like(game_meta["app_genres"]).value_counts()

taxonomy_profile = pd.DataFrame(
    {
        "metric": [
            "unique_games",
            "unique_app_categories",
            "unique_app_genres",
        ],
        "value": [
            game_meta["app_name"].nunique(),
            int(category_counts.index.nunique()),
            int(genre_counts.index.nunique()),
        ],
    }
)

display(dataset_profile)
display(taxonomy_profile)
display(category_counts.head(15).rename_axis("app_category").reset_index(name="game_count"))
display(genre_counts.head(15).rename_axis("app_genre").reset_index(name="game_count"))


,metric,value
0,review_votes_funny_min,0
1,review_votes_funny_max,3595
2,review_timestamp_min,2015-11-11 11:07:15+00:00
3,review_timestamp_max,2026-01-13 00:59:31+00:00
4,review_text_cleaned_chars_min,1
5,review_text_cleaned_chars_max,8000
6,review_text_cleaned_chars_median,73.0000
7,review_text_cleaned_words_min,1
8,review_text_cleaned_words_max,2588
9,review_text_cleaned_words_median,14.0000


,metric,value
0,unique_games,23
1,unique_app_categories,43
2,unique_app_genres,10


,app_category,game_count
0,Steam Achievements,22
1,Single-player,20
2,Full controller support,18
3,Family Sharing,16
4,Steam Cloud,16
5,Multi-player,16
6,Co-op,15
7,Online Co-op,13
8,Steam Trading Cards,12
9,Custom Volume Controls,10


,app_genre,game_count
0,Action,17
1,Adventure,9
2,RPG,8
3,Indie,7
4,Casual,5
5,Strategy,5
6,Simulation,4
7,Free To Play,1
8,Massively Multiplayer,1
9,Early Access,1


## 1) Data Quality and Coverage

In [ ]:
quality_cols = [
    "review_text_cleaned",
    "review_info_text_cleaned",
    "label_is_funny_binary",
    "label_funny_minmax",
    "label_votes_funny_categorical",
    "review_votes_funny",
    "review_votes_up",
    "author_playtime_forever",
    "app_release_date",
]

missing_df = (
    df[quality_cols]
    .isna()
    .mean()
    .mul(100)
    .rename("missing_pct")
    .sort_values(ascending=False)
    .rename_axis("column")
    .reset_index(name="missing_pct")
)

game_coverage = (
    df.groupby("app_name", observed=True)
    .agg(
        reviews=("review_id", "count"),
        funny_rate=("label_is_funny_binary", "mean"),
        median_votes_funny=("review_votes_funny", "median"),
        p95_votes_funny=("review_votes_funny", lambda s: s.quantile(0.95)),
    )
    .assign(funny_rate=lambda x: x["funny_rate"] * 100)
    .sort_values("reviews", ascending=False)
)

display(missing_df)
display(game_coverage.head(10).round(3))

plot_game = game_coverage.sort_values("funny_rate", ascending=False).reset_index()
fig, ax = plt.subplots(figsize=(12, 6))

sns.scatterplot(
    data=plot_game,
    x="funny_rate",
    y="reviews",
    size="reviews",
    sizes=(50, 800),
    hue="median_votes_funny",
    palette="viridis",
    ax=ax,
    legend=False,
)
for _, r in plot_game.iterrows():
    ax.text(r["funny_rate"] + 0.02, r["reviews"], str(r["app_name"]), fontsize=8)
ax.set_title("Game-Level Humor Rate vs Review Volume")
ax.set_xlabel("Funny-label rate (%)")
ax.set_ylabel("Review count")

plt.tight_layout()
plt.show()


## 2) Label Distributions and Imbalance

In [ ]:
cat_map = {
    0: "serious",
    1: "witty",
    2: "amusing",
    3: "hilarious",
    4: "hysterical",
}

decimal_cls = (df["label_funny_minmax"] * 10).round().astype("Int64")
decimal_cls = decimal_cls[decimal_cls.between(0, 9)]

def dist_table(series, label_map=None):
    counts = series.value_counts(dropna=False).sort_index()
    out = pd.DataFrame({"count": counts})
    out["pct"] = out["count"] / out["count"].sum() * 100
    out.index.name = "class"
    out = out.reset_index()
    if label_map is not None:
        out["label"] = out["class"].map(label_map)
    return out

cat_dist = dist_table(df["label_votes_funny_categorical"].astype("Int64"), cat_map)
decimal_dist = dist_table(decimal_cls)
decimal_dist["label"] = decimal_dist["class"].apply(lambda x: f"0.{x}")

display(cat_dist)
display(decimal_dist)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=cat_dist, x="label", y="count", ax=axes[0], color="#55A868")
axes[0].set_title("Categorical Humor Label Distribution")
axes[0].set_xlabel("")
axes[0].set_ylabel("Reviews")
axes[0].tick_params(axis="x", rotation=25)
axes[0].set_yscale("log")

sns.barplot(data=decimal_dist, x="label", y="count", ax=axes[1], color="#C44E52")
axes[1].set_title("Decimal Humor Label Distribution")
axes[1].set_xlabel("")
axes[1].set_ylabel("Reviews")
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()


## 3) Prompt-Level Balanced-Sampling Capacity

In [ ]:
def class_ids_for_prompt(prompt_spec, frame):
    label_col = prompt_spec["label_col"]
    if label_col == "label_funny_minmax":
        cls = (frame[label_col].astype(float) * 10).round().astype("Int64")
    else:
        cls = frame[label_col].astype("Int64")
    return cls[cls.between(0, len(prompt_spec["label_words"]) - 1)]


ZERO_CLASS_TO_NONZERO_RATIO = 1.0
MAX_ZERO_CLASS_SAMPLES = None

rows = []
for spec in PROMPT_SPECS:
    cls = class_ids_for_prompt(spec, df)
    counts = cls.value_counts().sort_index()
    classes = len(spec["label_words"])

    if counts.empty:
        continue

    zero_count = int(counts.get(0, 0))
    nonzero_counts = counts[counts.index != 0]
    nonzero_total = int(nonzero_counts.sum())
    min_nonzero_count = int(nonzero_counts.min()) if len(nonzero_counts) else 0
    max_nonzero_count = int(nonzero_counts.max()) if len(nonzero_counts) else 0

    class0_target = int(round(nonzero_total * ZERO_CLASS_TO_NONZERO_RATIO))
    class0_target = max(1, class0_target) if zero_count > 0 else 0
    if MAX_ZERO_CLASS_SAMPLES is not None:
        class0_target = min(class0_target, int(MAX_ZERO_CLASS_SAMPLES))
    class0_used = min(zero_count, class0_target)

    sampled_total = class0_used + nonzero_total
    dropped_total = int(counts.sum()) - sampled_total

    rows.append(
        {
            "prompt_id": spec["id"],
            "label_col": spec["label_col"],
            "num_classes": classes,
            "class0_count_before": zero_count,
            "class0_count_used": class0_used,
            "nonzero_total": nonzero_total,
            "min_nonzero_class_count": min_nonzero_count,
            "max_nonzero_class_count": max_nonzero_count,
            "nonzero_imbalance_max_over_min": (
                round(max_nonzero_count / max(min_nonzero_count, 1), 2)
                if min_nonzero_count > 0
                else np.nan
            ),
            "sampled_total_rows": sampled_total,
            "dropped_rows": dropped_total,
            "drop_rate_pct": 100.0 * dropped_total / max(int(counts.sum()), 1),
            "prompt_word_count": len(spec["prompt"].split()),
        }
    )

capacity_df = pd.DataFrame(rows).sort_values(["sampled_total_rows", "prompt_id"], ascending=[False, True])
display(capacity_df)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=capacity_df, x="prompt_id", y="sampled_total_rows", color="#8172B3", ax=ax)
ax.set_title("Usable Samples per Prompt (Downsample Class 0 Only)")
ax.set_xlabel("Prompt ID")
ax.set_ylabel("Sampled rows")
for i, r in capacity_df.reset_index(drop=True).iterrows():
    ax.text(
        i,
        r["sampled_total_rows"] + max(capacity_df["sampled_total_rows"]) * 0.01,
        f"{int(r['sampled_total_rows']):,}",
        ha="center",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

print(
    "Note: Possible future variant for decimal prompts: class 0 for votes==0 and "
    "non-zero classes from log1p(review_votes_funny) quantile bins."
)


## 4) Input Length and Truncation Risk (BERT MLM Setup)

In [ ]:
hf_logging.set_verbosity_error()

try:
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
except Exception:
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

pair_special_tokens = tokenizer.num_special_tokens_to_add(pair=True)  # [CLS], [SEP], [SEP]


def token_lengths(text_series: pd.Series, batch_size: int = 4096) -> np.ndarray:
    text_series = text_series.fillna("").astype(str).reset_index(drop=True)
    lengths = np.empty(len(text_series), dtype=np.int32)
    for start in range(0, len(text_series), batch_size):
        end = min(start + batch_size, len(text_series))
        batch = text_series.iloc[start:end].tolist()
        enc = tokenizer(
            batch,
            add_special_tokens=False,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False,
        )
        lengths[start:end] = [len(ids) for ids in enc["input_ids"]]
    return lengths

lengths_by_input = {
    name: token_lengths(df[col])
    for name, col in TEXT_COLUMNS.items()
}

for name, arr in lengths_by_input.items():
    df[f"{name}_token_len"] = arr

length_summary = (
    pd.DataFrame(
        {
            name: pd.Series(arr)
            for name, arr in lengths_by_input.items()
        }
    )
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .T
)

display(length_summary)

trunc_rows = []
for spec in PROMPT_SPECS:
    prompt_tokens = len(tokenizer.tokenize(spec["prompt"]))
    allowed_text_tokens = MAX_LENGTH - prompt_tokens - pair_special_tokens
    for input_name, arr in lengths_by_input.items():
        truncated = arr > allowed_text_tokens
        kept = np.minimum(arr, allowed_text_tokens)
        trunc_rows.append(
            {
                "prompt_id": spec["id"],
                "input_type": input_name,
                "prompt_token_len": prompt_tokens,
                "allowed_text_tokens": allowed_text_tokens,
                "truncation_rate_pct": truncated.mean() * 100,
                "mean_kept_token_pct": (kept / np.maximum(arr, 1)).mean() * 100,
                "p95_tokens_lost_if_truncated": np.quantile(np.maximum(arr - allowed_text_tokens, 0), 0.95),
            }
        )

trunc_df = pd.DataFrame(trunc_rows).sort_values(["input_type", "prompt_id"])
display(trunc_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

heat = trunc_df.pivot(index="prompt_id", columns="input_type", values="truncation_rate_pct")
sns.heatmap(heat, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[0])
axes[0].set_title("Truncation Rate (%) at max_length=256")

plot_df = pd.concat(
    [
        pd.DataFrame({"token_len": arr, "input_type": name})
        for name, arr in lengths_by_input.items()
    ],
    ignore_index=True,
)

sample_n = min(120_000, len(plot_df))
plot_sample = plot_df.sample(n=sample_n, random_state=42)
sns.kdeplot(data=plot_sample, x="token_len", hue="input_type", common_norm=False, fill=True, alpha=0.2, ax=axes[1])
axes[1].set_xlim(0, np.quantile(plot_sample["token_len"], 0.995))
axes[1].set_title("Token-Length Density by Input Type")
axes[1].set_xlabel("BERT wordpiece length (text only)")

plt.tight_layout()
plt.show()


## 5) Temporal Behavior and Vote Signals

In [ ]:
sample = df.sample(n=min(120_000, len(df)), random_state=42).copy()
sample["log_votes_up"] = np.log1p(sample["review_votes_up"])
sample["log_votes_funny"] = np.log1p(sample["review_votes_funny"])

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.histplot(
    data=sample,
    x="log_votes_funny",
    bins=60,
    element="step",
    stat="density",
    ax=axes[0],
)
axes[0].set_title("Log Funny Votes Distribution")
axes[0].set_xlabel("log(1 + votes_funny)")

hb = axes[1].hexbin(sample["log_votes_up"], sample["log_votes_funny"], gridsize=55, cmap="viridis", mincnt=1)
axes[1].set_title("Votes Up vs Votes Funny (Hexbin, log scale)")
axes[1].set_xlabel("log(1 + votes_up)")
axes[1].set_ylabel("log(1 + votes_funny)")
fig.colorbar(hb, ax=axes[1], label="count")

plt.tight_layout()
plt.show()


## 6) Feature Correlation and Label-Word Leakage Checks

In [ ]:
numeric_cols = [
    "review_votes_funny",
    "review_votes_up",
    "review_weighted_vote_score",
    "review_comment_count",
    "author_playtime_forever",
    "author_playtime_at_review",
    "review_only_token_len",
    "review_plus_game_token_len",
    "label_is_funny_binary",
    "label_funny_minmax",
    "label_votes_funny_categorical",
]

corr = df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Matrix (Numeric Features and Labels)")
plt.tight_layout()
plt.show()

label_words = ["serious", "witty", "amusing", "hilarious", "hysterical"]

leak_rows = []
for word in label_words:
    pattern = rf"\b{word}\b"
    for input_name, col in TEXT_COLUMNS.items():
        contains = df[col].fillna("").str.contains(pattern, case=False, regex=True)
        leak_rows.append(
            {
                "word": word,
                "input_type": input_name,
                "contains_rate_pct": contains.mean() * 100,
                "contains_rate_funny_pct": contains[df["label_is_funny_binary"] == 1].mean() * 100,
                "contains_rate_not_funny_pct": contains[df["label_is_funny_binary"] == 0].mean() * 100,
            }
        )

leak_df = pd.DataFrame(leak_rows).sort_values(["word", "input_type"])
display(leak_df.round(4))

plt.figure(figsize=(12, 5))
sns.barplot(data=leak_df, x="word", y="contains_rate_pct", hue="input_type", palette="Set2")
plt.title("Frequency of Prompt Label Words in Input Texts")
plt.xlabel("Label word")
plt.ylabel("Reviews containing word (%)")
plt.tight_layout()
plt.show()


## 7) What This Means for the Seminar Experiments

Use the tables/plots above directly in the paper to justify:

- Why decimal prompts are harder to train robustly (`label_funny_minmax` has extreme tail sparsity).
- Why `review_info_text_cleaned` can change performance (richer context but higher truncation pressure).
- Why macro-F1 should be emphasized over raw accuracy for imbalanced label spaces.
- Why prompt wording effects can be partially explained by token budget differences and lexical priors.


## 8) Sequence Length Tradeoff (Coverage vs Compute)

This section estimates how often reviews are truncated for different `MAX_LENGTH` values when we encode as a text pair:

`[CLS] review_text [SEP] prompt [SEP]`

So available review tokens are:

`max_review_budget = MAX_LENGTH - prompt_token_len - 3`

It reports truncation/keep rates across all seminar prompts and gives practical recommendations per text column.


In [ ]:
# Reuse notebook-level constants from the setup section.
PROMPTS_FOR_LENGTH_CHECK = [spec["prompt"] for spec in PROMPT_SPECS]
TEXT_COLUMNS_FOR_LENGTH_CHECK = TEXT_COLUMNS.copy()
MAX_LENGTH_CANDIDATES = sorted(set([128, 192, 256, 320, 384, 448, 490, 512, MAX_LENGTH]))


pair_special_tokens_for_length = (
    pair_special_tokens
    if "pair_special_tokens" in globals()
    else tokenizer.num_special_tokens_to_add(pair=True)
)

prompt_token_lens = {
    spec["id"]: len(tokenizer.encode(spec["prompt"], add_special_tokens=False))
    for spec in PROMPT_SPECS
}

print("Prompt token lengths:")
for spec in PROMPT_SPECS:
    pid = spec["id"]
    print(f"- {pid:9s}: {prompt_token_lens[pid]:2d} :: {spec['prompt']}")



In [ ]:
length_analysis_rows = []

for input_name, text_col in TEXT_COLUMNS_FOR_LENGTH_CHECK.items():
    series = df[text_col].dropna().astype(str)

    # Token length distribution for the raw text column
    text_lens = series.apply(
        lambda x: len(tokenizer.encode(x, add_special_tokens=False))
    ).to_numpy()

    quantiles = np.quantile(text_lens, [0.5, 0.75, 0.9, 0.95, 0.99])
    print("=" * 90)
    print(f"{input_name} ({text_col})")
    print(
        "token quantiles: "
        f"p50={quantiles[0]:.1f}, p75={quantiles[1]:.1f}, p90={quantiles[2]:.1f}, "
        f"p95={quantiles[3]:.1f}, p99={quantiles[4]:.1f}"
    )

    for max_length in MAX_LENGTH_CANDIDATES:
        # Pair encoding budget: [CLS] text [SEP] prompt [SEP]
        review_budgets = {
            pid: (max_length - p_len - pair_special_tokens_for_length)
            for pid, p_len in prompt_token_lens.items()
        }

        trunc_rates = {
            pid: float((text_lens > budget).mean())
            for pid, budget in review_budgets.items()
        }

        keep_rates = {pid: 1.0 - r for pid, r in trunc_rates.items()}

        length_analysis_rows.append(
            {
                "input_name": input_name,
                "text_col": text_col,
                "max_length": max_length,
                "min_review_budget": min(review_budgets.values()),
                "worst_keep_rate": min(keep_rates.values()),
                "mean_keep_rate": float(np.mean(list(keep_rates.values()))),
                "worst_trunc_rate": max(trunc_rates.values()),
                "mean_trunc_rate": float(np.mean(list(trunc_rates.values()))),
                # Rough attention complexity proxy
                "rel_attention_cost_vs_256": (max_length / 256.0) ** 2,
            }
        )

length_analysis_df = pd.DataFrame(length_analysis_rows)

for input_name, text_col in TEXT_COLUMNS_FOR_LENGTH_CHECK.items():
    print("\n" + "-" * 90)
    print(f"Tradeoff table: {input_name} ({text_col})")
    sub = length_analysis_df[length_analysis_df["input_name"] == input_name].copy()

    display(
        sub[
            [
                "max_length",
                "min_review_budget",
                "worst_keep_rate",
                "mean_keep_rate",
                "worst_trunc_rate",
                "mean_trunc_rate",
                "rel_attention_cost_vs_256",
            ]
        ]
        .sort_values("max_length")
        .round(4)
    )

    # Recommendations by keep-rate threshold
    for threshold in [0.95, 0.97]:
        candidates = sub[sub["worst_keep_rate"] >= threshold].sort_values(
            ["rel_attention_cost_vs_256", "max_length"]
        )
        if len(candidates) > 0:
            pick = candidates.iloc[0]
            print(
                f"Recommended for >= {threshold:.0%} worst-case keep: "
                f"MAX_LENGTH={int(pick['max_length'])} "
                f"(cost x{pick['rel_attention_cost_vs_256']:.2f} vs 256)"
            )
        else:
            print(f"No candidate reaches >= {threshold:.0%} worst-case keep.")



## 9) Max Sample Size Choice (Runtime-Aware)

This section helps pick `MAX_TOTAL_SAMPLES_PER_PROMPT` for fine-tuning.

It uses the already computed `capacity_df` (after downsampling class 0 only), then simulates different hard caps and reports:
- rows kept
- estimated train rows
- estimated steps per epoch (`ceil(train_rows / batch_size)`)
- relative compute proxy vs full prompt data (`steps * max_length^2`)


In [ ]:
# Runtime/cap analysis assumptions (edit to match your run settings)
EST_BATCH_SIZE = 16
EST_MAX_LENGTH = 490
CAP_CANDIDATES = [5000, 8000, 10000, 12000, 15000, 20000, 30000, 40000, 60000, 80000, 100000, None]

if "capacity_df" not in globals() or capacity_df.empty:
    raise ValueError("Run the sampling-capacity cell first so `capacity_df` exists.")

rows = []
for _, r in capacity_df.iterrows():
    prompt_id = r["prompt_id"]
    full_rows = int(r["sampled_total_rows"])

    for cap in CAP_CANDIDATES:
        if cap is None:
            effective_rows = full_rows
            cap_label = "full"
            cap_value = full_rows
        else:
            cap = int(cap)
            effective_rows = min(full_rows, cap)
            cap_label = str(cap)
            cap_value = cap

        train_rows = int(np.floor(effective_rows * 0.8))
        steps_per_epoch = int(np.ceil(train_rows / EST_BATCH_SIZE))

        rows.append(
            {
                "prompt_id": prompt_id,
                "label_col": r["label_col"],
                "full_rows": full_rows,
                "cap_label": cap_label,
                "cap_value": cap_value,
                "effective_rows": effective_rows,
                "retention_pct": 100.0 * effective_rows / max(full_rows, 1),
                "train_rows_est": train_rows,
                "steps_per_epoch_est": steps_per_epoch,
                "epoch_cost_proxy": float(steps_per_epoch * (EST_MAX_LENGTH ** 2)),
            }
        )

cap_df = pd.DataFrame(rows)

# Relative cost vs using full rows for that same prompt
full_cost = (
    cap_df[cap_df["cap_label"] == "full"]
    [["prompt_id", "epoch_cost_proxy"]]
    .rename(columns={"epoch_cost_proxy": "full_epoch_cost_proxy"})
)
cap_df = cap_df.merge(full_cost, on="prompt_id", how="left")
cap_df["epoch_cost_rel_to_full_pct"] = 100.0 * cap_df["epoch_cost_proxy"] / cap_df["full_epoch_cost_proxy"]

# Focus plot on common seminar pair if present; otherwise show all prompts
preferred = ["decimal_0", "words_0"]
prompts_to_plot = [p for p in preferred if p in cap_df["prompt_id"].unique()]
if len(prompts_to_plot) == 0:
    prompts_to_plot = sorted(cap_df["prompt_id"].unique())

plot_df = cap_df[cap_df["prompt_id"].isin(prompts_to_plot)].copy()
plot_df = plot_df.sort_values(["prompt_id", "effective_rows"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(
    data=plot_df,
    x="effective_rows",
    y="steps_per_epoch_est",
    hue="prompt_id",
    marker="o",
    ax=axes[0],
)
axes[0].set_title(f"Estimated Steps/Epoch vs Kept Rows (batch={EST_BATCH_SIZE})")
axes[0].set_xlabel("Kept rows after cap")
axes[0].set_ylabel("Estimated steps per epoch")

sns.lineplot(
    data=plot_df,
    x="effective_rows",
    y="epoch_cost_rel_to_full_pct",
    hue="prompt_id",
    marker="o",
    ax=axes[1],
    legend=False,
)
axes[1].set_title(f"Relative Epoch Compute vs Full (max_length={EST_MAX_LENGTH})")
axes[1].set_xlabel("Kept rows after cap")
axes[1].set_ylabel("Compute proxy (% of full prompt data)")

for x in [10000, 15000, 20000]:
    axes[0].axvline(x=x, color="gray", linestyle="--", alpha=0.3)
    axes[1].axvline(x=x, color="gray", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

# Candidate table for words_0 (or largest prompt by rows if words_0 unavailable)
focus_prompt = "words_0" if "words_0" in cap_df["prompt_id"].unique() else cap_df.groupby("prompt_id")["full_rows"].max().idxmax()
focus = cap_df[cap_df["prompt_id"] == focus_prompt].copy()
focus = focus.sort_values(["effective_rows", "cap_value"])

print(f"Candidate caps for prompt: {focus_prompt}")
display(
    focus[
        [
            "cap_label",
            "effective_rows",
            "retention_pct",
            "train_rows_est",
            "steps_per_epoch_est",
            "epoch_cost_rel_to_full_pct",
        ]
    ].drop_duplicates(subset=["cap_label", "effective_rows"]).round(2)
)
